# Data Cleaning

## Importing

In [ ]:
#for cleaning
import pandas as pd
import numpy as np
#for analysis
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore") #Some analysis had problems with warnings

## Common Data Set \(CDS\) Scraping

The CDS contains data about 2025\-26 applications that could not be found in the IPEDS dataset\. 2025\-26 is an important year to include as a post\-AI datapoint since AI is very easily accessible and at its most accurate during this time\. The terms and conditions of the website were read to assure that scraping of the website was allowed\. After all the 2025\-26 data on all universities present on the website was scraped, it was put into commondatasets\_admissions\.csv \.

The dataframe can be cleaned to only contain the data needed about applications received by each university, amount of applications admitted, and school information\. Before removing unwanted columns, the dataframe must be read and displayed so that we can eliminate columns unrelated to our question\.

In [ ]:
df = pd.read_csv("commondatasets_admissions.csv")
[c for c in df.columns if any(k in c.lower() for k in ["appl", "admit", "name", "school"])]

['name',
 'school_type',
 'applicants_total',
 'applicants_men',
 'applicants_women',
 'admitted_total',
 'admitted_men',
 'admitted_women',
 'application_fee',
 'waitlist_admitted',
 'ea_apps_admitted',
 'applicant_pools.ea',
 'applicant_pools.rd',
 'applicant_pools.waitlist',
 'gender_breakdown.applied.male',
 'gender_breakdown.applied.female',
 'admission_factors.applicant_interest',
 'transfer_stats.male.applied',
 'transfer_stats.male.admitted',
 'transfer_stats.female.applied',
 'transfer_stats.female.admitted',
 'transfer_stats.total.applied',
 'transfer_stats.total.admitted',
 'applicant_pools',
 'applicant_pools.ea.applied',
 'applicant_pools.ea.admitted',
 'applicant_pools.ea.rate',
 'applicant_pools.rd.applied',
 'applicant_pools.rd.admitted',
 'applicant_pools.rd.rate',
 'applicant_pools.waitlist.offered',
 'applicant_pools.waitlist.accepted_spots',
 'applicant_pools.waitlist.enrolled']

Now, we only keep the necessary columns to our research question about applications sent out and accepted, and school information\. Also, the acceptance rate has to be calculated manually by dividing total admitted students for a school by applicants total\. Since all this data is from 2025\-26, the year for each datapoint is chosen to be 2025\.

In [ ]:
latest_df_clean=df[["name","applicants_total","admitted_total"]].copy()
latest_df_clean["acceptance_rate"]=latest_df_clean["admitted_total"]/latest_df_clean["applicants_total"]
latest_df_clean["year"]=2025
latest_df_clean.to_csv("clean_df_2025.csv", index=False)
latest_df_clean.head(10)

,name,applicants_total,admitted_total,acceptance_rate,year
0,University of Alabama at Birmingham,11119,10474,0.941991,2025
1,American University,20960,13741,0.655582,2025
2,Amherst College,15819,1222,0.077249,2025
3,Arizona State University,69617,61533,0.883879,2025
4,Auburn University,59121,27724,0.468937,2025
5,Barnard College,10426,1048,0.100518,2025
6,Baylor University,44924,23459,0.522193,2025
7,UC Berkeley,126864,14524,0.114485,2025
8,Binghamton University,61197,22835,0.373139,2025
9,Boston University,76776,9853,0.128334,2025


## IPEDS Dataset

Open all of the datasets in the folder "admissionenroll1724", which contains all admission data available for universities present in the database from 2017 to 2024\. From IPEDS there were datasets for every single year\. These were added to a folder which was uploaded into the deepnote project\. In order to clean up the data, this code has to open all of the files\.

In [ ]:
datasets = []
for year in range(2017, 2025):
    df = pd.read_csv('admissionenroll1724/adm_' + str(year) + '.csv', encoding='utf-8-sig')
    datasets.append(df)

len(datasets)

8

Begin creating a combined table that will have all of the years in one table\. All of the years are separate files on the IPEDS website\. In order to conduct the calculations and analysis, in addition to just having it more visually pleasing for reading the data, the years needed to be combined into one dataset\. This code block adds all years from 2017\-2024, downloaded from IPEDS, into one dataset\.

In [ ]:
for df in datasets:
    df.columns = [col.split('.')[-1] for col in df.columns]

combined = pd.concat(datasets)

combined.shape

(47768, 12)

Drop the undesired columns\. Not all of the columns in the admissions data are useful to us\. When downloading from the IPEDS website, the admissions section contains lots of information beyond just acceptance rates and applications submitted\. So, we entered in the desired columns and had all of the columns which did not match our desired columns dropped\. This cleans up the data a lot, meaning the information is all informative for our project now and there is a lot less raw data that we have to sift through\. The files also get significantly smaller\.

In [ ]:
combined = combined[['unitid', 'institution name', 'year',
                     'Applicants total', 'Admissions total', 'Enrolled total']]
combined.columns = ['unitid', 'name', 'year', 'applicants_total', 'admitted_total', 'enrolled_total']

combined = combined.dropna(subset=['applicants_total'])

combined.shape

(15258, 6)

Reorder so that it is organized by school name, not by year\. This utilizes a sorting chunk of code\. In order to be able to read the raw data easier, we believed that seeing it ordered by school rather than by year would make for more digestible code\. So, a reordering code chunk was written\.

In [ ]:
combined = combined.sort_values(['name', 'unitid', 'year']).reset_index(drop=True)

combined.to_csv('ipeds_2017_2024.csv', index=False)

combined.head(10)

,unitid,name,year,applicants_total,admitted_total,enrolled_total
0,214023,ASPIRA City College,2017,3.0,3.0,2.0
1,214023,ASPIRA City College,2018,8.0,4.0,2.0
2,214023,ASPIRA City College,2021,3.0,3.0,3.0
3,214023,ASPIRA City College,2022,3.0,3.0,2.0
4,214023,ASPIRA City College,2023,3.0,3.0,3.0
5,214023,ASPIRA City College,2024,6.0,3.0,3.0
6,222178,Abilene Christian University,2017,10252.0,5952.0,972.0
7,222178,Abilene Christian University,2018,11569.0,6590.0,924.0
8,222178,Abilene Christian University,2019,11379.0,6996.0,932.0
9,222178,Abilene Christian University,2020,10534.0,6679.0,859.0


Remove entries if there were under 100 applications submitted to said college in a given year\. This is done through a simple piece of code which reads through every single year for a school\. If in any given year the amount of applications received was under 100, then all entries for that specific school are going to be removed\. We believe that in order for the data to be representative of overall trends, there needs to be a baseline number of applications, 100, that a school was consistently receiving\.

In [ ]:
combined = combined[~combined['unitid'].isin(combined[combined['applicants_total'] < 100]['unitid'])].reset_index(drop=True)
combined.to_csv('ipeds_2017_2024.csv', index=False)

combined.head(10)

,unitid,name,year,applicants_total,admitted_total,enrolled_total
0,222178,Abilene Christian University,2017,10252.0,5952.0,972.0
1,222178,Abilene Christian University,2018,11569.0,6590.0,924.0
2,222178,Abilene Christian University,2019,11379.0,6996.0,932.0
3,222178,Abilene Christian University,2020,10534.0,6679.0,859.0
4,222178,Abilene Christian University,2021,8208.0,5755.0,833.0
5,222178,Abilene Christian University,2022,11226.0,7386.0,893.0
6,222178,Abilene Christian University,2023,10510.0,6714.0,862.0
7,222178,Abilene Christian University,2024,11332.0,7482.0,906.0
8,138558,Abraham Baldwin Agricultural College,2017,1927.0,1353.0,883.0
9,138558,Abraham Baldwin Agricultural College,2018,2389.0,1647.0,1017.0


Add the acceptance rate and yield rate for every school\. Acceptance rate and yield rate are important measures which can be evaluated for further analysis of the trends in college admissions during the baseline, COVID, and AI eras\. These numbers were already tracked in the original dataset\. So, they are brought back and amended onto the data once more\.

In [ ]:
combined['acceptance_rate'] = (combined['admitted_total'] / combined['applicants_total'] * 100).round(1)
combined['yield_rate'] = (combined['enrolled_total'] / combined['admitted_total'] * 100).round(1)

combined.to_csv('ipeds_2017_2024.csv', index=False)

combined.head(10)

,unitid,name,year,applicants_total,admitted_total,enrolled_total,acceptance_rate,yield_rate
0,222178,Abilene Christian University,2017,10252.0,5952.0,972.0,58.1,16.3
1,222178,Abilene Christian University,2018,11569.0,6590.0,924.0,57.0,14.0
2,222178,Abilene Christian University,2019,11379.0,6996.0,932.0,61.5,13.3
3,222178,Abilene Christian University,2020,10534.0,6679.0,859.0,63.4,12.9
4,222178,Abilene Christian University,2021,8208.0,5755.0,833.0,70.1,14.5
5,222178,Abilene Christian University,2022,11226.0,7386.0,893.0,65.8,12.1
6,222178,Abilene Christian University,2023,10510.0,6714.0,862.0,63.9,12.8
7,222178,Abilene Christian University,2024,11332.0,7482.0,906.0,66.0,12.1
8,138558,Abraham Baldwin Agricultural College,2017,1927.0,1353.0,883.0,70.2,65.3
9,138558,Abraham Baldwin Agricultural College,2018,2389.0,1647.0,1017.0,68.9,61.7


## Combine datasets

Now that we have two separate datasets, we need to combine them to get a full picture of the college application system before and after the rise of AI\. We need to only include schools present in both dataframes since we want a full picture of admissions data from 2017 to 2025\. Then, we can do analysis on the finished dataset\.

In [ ]:
# Schools present in both datasets
common = set(latest_df_clean["name"]) & set(combined["name"])
print(len(common), "schools appear in both")

# Keep only the shared schools in each, then stack
final = pd.concat([
    latest_df_clean[latest_df_clean["name"].isin(common)],
    combined[combined["name"].isin(common)],
], ignore_index=True)

#Organize by school name
final = final.sort_values(by="name")
final.to_csv("final_df.csv", index=False)

# Check: dataframe shape
print(final.shape)
final.head()

123 schools appear in both
(1107, 8)


,name,applicants_total,admitted_total,acceptance_rate,year,unitid,enrolled_total,yield_rate
129,American University,17786.0,8427.0,47.400000,2023,131159.0,1856.0,22.0
1,American University,20960.0,13741.0,0.655582,2025,NaN,NaN,NaN
130,American University,17173.0,10640.0,62.000000,2024,131159.0,1662.0,15.6
128,American University,19510.0,7926.0,40.600000,2022,131159.0,1761.0,22.2
127,American University,19650.0,12594.0,64.100000,2021,131159.0,2372.0,18.8


One issue presented here is that only the IPEDS dataset contained information about enrollment and individual applicant information\. Since this data could be valuable, we decided to keep the data and allow the 2025 data to be NaN for these columns\. If it provides a meaningful output, this IPEDS data will be used and have the 2025 datapoints omitted in the future\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5504e1cb-e852-40a9-a959-1b5368c81868' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>